# BNIA Vital Signs 2010–2023 — Cleaning, EDA & Tableau Prep
### The Lily Foundation & Advocates

Raw Data → **Cleaning** → **Analysis** → Visualization (Tableau) → Interpretation → Recommendation

This notebook covers the first two stages and exports Tableau-ready files.
**No charts are produced here** — visuals are built in Tableau.

---

### What this workbook actually contains

56 rows per sheet (55 Community Statistical Areas + 1 `Baltimore City` aggregate),
14 year sheets, 223 distinct indicators. The real header is **row 2**; row 1 is a
column-number ruler.

### Answering "do I use all rows or just Baltimore City?"

**All 55 CSA rows.** The assignment is about disparities *between* communities, and a
single citywide row has no variance to analyse. It is also not neutral to leave in —
it sits near the population-weighted middle and drags every z-score, correlation and
quartile break.

But it is not discarded either. It becomes the **benchmark** — the reference line on
every Tableau chart that lets a viewer see instantly whether a community sits above or
below the city as a whole. It is exported separately as `citywide_benchmark.csv`.

### The constraint that shapes the whole analysis

Health outcomes and current socioeconomic data **never coexist** in this file:

| Indicator family | Years available |
|---|---|
| Life expectancy, mortality by age | 2011–2018 |
| Birth outcomes, prenatal care, child lead | 2010–2019 |
| Income, poverty, vehicles, internet, education | through 2023 |
| Health insurance, HIV, causes of death, EMS response | **empty in all 14 years** |

There is no year after 2019 containing a single health outcome. Section 6 quantifies
this and picks the cross-section year accordingly. There are also **no emergency-
department indicators** anywhere in this workbook.

## 0. Setup

In [1]:
import os, re, glob, warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

# Point this at your workbook. Falls back to the first matching file nearby.
# The workbook ships in raw/ and uses spaces, not underscores, in its name.
XLSX = "raw/Vital Signs 2010-2023.xlsx"
if not os.path.exists(XLSX):
    hits = (glob.glob("raw/*Vital*Signs*.xls*")
            + glob.glob("*Vital*Signs*.xls*")
            + glob.glob("*/*Vital*Signs*.xls*")
            + glob.glob("/mnt/user-data/uploads/*Vital*Signs*.xls*"))
    if hits:
        XLSX = hits[0]
    else:
        raise FileNotFoundError(
            "Vital Signs workbook not found. Place it in raw/ or set XLSX manually.")

OUT = "tableau"
YEARS = list(range(2010, 2024))
HEADER_ROW = 1            # row 0 is a column-number ruler, not data
CITY_ROW = "Baltimore City"

os.makedirs(OUT, exist_ok=True)
print("Workbook:", XLSX)
print("Exists:  ", os.path.exists(XLSX))

log = []
def note(msg):
    '''Record a cleaning decision for the methodology log the brief asks for.'''
    log.append(msg); print("  NOTE:", msg)

Workbook: raw/Vital Signs 2010-2023.xlsx
Exists:   True


## 1. Helpers

In [2]:
def norm_name(c):
    """Collapse whitespace and trim so the same indicator matches across years.

    The workbook renames indicators between sheets in purely cosmetic ways:
    'Percent of Population 18-24 Years old' (2010) vs '... 18-24 years old'
    (2023), and a double space in 'Population  Under 5'. Concatenating without
    this creates duplicate columns that look like distinct indicators.
    """
    return re.sub(r"\s+", " ", str(c)).strip()

def key_name(c):
    return norm_name(c).lower()


TOPICS = [
    ("Health",       ["life expectancy", "mortality", "birth", "prenatal",
                      "lead levels", "teen pregnancy", "hiv", "death",
                      "health insurance", "fetal"]),
    # Income BEFORE Age & Family: six income bands are phrased "Percent of Households
    # Earning ..." and the keyword "households" would otherwise claim them first.
    ("Income",       ["income", "poverty", "earning", "tanf", "snap"]),
    ("Age & Family", ["years old", "years and over", "age", "household size",
                      "children under 18", "female-headed", "households"]),
    ("Employment",   ["employ", "unemploy", "labor force", "workforce",
                      "16-19 in school"]),
    ("Education",    ["student", "school", "diploma", "bachelor", "grade",
                      "dropout", "completion", "parcc", "msa ", "h.s.a"]),
    ("Housing",      ["housing", "properties", "vacant", "owner-occupied",
                      "foreclosure", "mortgage", "rent", "affordability",
                      "sales", "permits", "residential", "lien"]),
    ("Access",       ["vehicle", "internet", "travel time", "walk", "commut",
                      "drove alone", "carpool", "transit", "library"]),
    ("Safety",       ["crime", "arrest", "shooting", "homicide", "juvenile",
                      "street light", "car crash", "fire"]),
    ("Environment",  ["dirty streets", "storm drain", "tree", "open space",
                      "food", "recycl", "waste"]),
    ("Business",     ["business", "commercial", "naics", "arts-related"]),
    ("Civic",        ["vote", "registered", "census", "nonprofit"]),
]

# Keyword matching gets ~84% right; these 37 are the tail it cannot reach, so they are
# mapped explicitly rather than by bending keywords until they happen to fit.
TOPIC_OVERRIDES = {
    # Demographics -- population counts and race composition are context, not a topic
    "Total Population": "Demographics", "Total Male Population": "Demographics",
    "Total Female Population": "Demographics", "Racial Diversity Index": "Demographics",
    "Percent of Residents - Black/African-American (Non-Hispanic)": "Demographics",
    "Percent of Residents - White/Caucasian (Non-Hispanic)": "Demographics",
    "Percent of Residents - Hispanic": "Demographics",
    "Percent of Residents - Asian (Non-Hispanic)": "Demographics",
    "Percent of Residents - Two or More Races (Non-Hispanic)": "Demographics",
    "Percent of Residents - All Other Races (Hawaiian/ Pacific Islander, Alaskan/ "
    "Native American Other Race) (Non-Hispanic)": "Demographics",
    # Health -- substance-use burden and the food/alcohol environment
    "Number of Overdose Calls for Service per 1,000 Residents": "Health",
    "Number of Narcotics Calls for Service per 1,000 Residents": "Health",
    "Liquor Outlet density (per 1,000 Residents)": "Health",
    # Safety
    "Domestic Violence Calls For Service per 1,000 Residents": "Safety",
    "Number of Common Assault Calls for Service per 1,000 Residents": "Safety",
    "Number of Automobile Accident Calls for Service per 1,000 Residents": "Safety",
    "Percent of Adult Population on Parole/Probation": "Safety",
    # Income support
    "Percent of Families Receiving TANF": "Income",
    "Percent of Persons receiving SNAP": "Income",
    # Housing market
    "Median Price of Homes Sold": "Housing", "Number of Homes Sold": "Housing",
    "Median Number of Days on the Market": "Housing",
    "Rate of Evictions per 1,000 Residents": "Housing",
    # Access / mobility -- the first two are phrased "Percent of Households ...",
    # so the Age & Family keyword "households" claims them without this override.
    "Percent of Households with No Vehicles Available": "Access",
    "Percent of Households with No Internet at Home": "Access",
    "Percent of Population that Uses Public Transportation to Get to Work": "Access",
    "Number of daily bus stop loadings per 1,000 residents": "Access",
    "Number of Miles of Bike Lanes": "Access",
    "Percent of the Population that Works from Home": "Access",
    # Environment / utilities
    "Median Daily Water Consumption": "Environment",
    "Percent of Homes Weatherized": "Environment",
    "Percent of Residences Heated by Electricity": "Environment",
    "Percent of Residences Heated by Utility Gas": "Environment",
    # Business & civic
    "Economic Diversity Index": "Business", "Number of Total Jobs": "Business",
    "Number of Banks and Bank Branches per 1,000 Residents": "Business",
    "Number of Public Murals": "Civic", "Public Art per 1,000 Residents": "Civic",
    # Education
    "Kindergarten Readiness": "Education",
}

def topic_of(name):
    if name in TOPIC_OVERRIDES:
        return TOPIC_OVERRIDES[name]
    n = name.lower()
    for topic, kws in TOPICS:
        # \b before each keyword so "age" does not match "mortgage", "percentage" or
        # "average" -- that mis-filed 40 indicators (every PARCC/MSA score, most of
        # Housing) into Age & Family. Prefix matches like "employ" -> "employment" still work.
        if any(re.search(r"\b" + re.escape(k), n) for k in kws):
            return topic
    return "Other"

def unit_of(name):
    n = name.lower()
    if n.startswith("percent") or "percentage" in n or n.startswith("%"):
        return "percent"
    if "rate" in n or "per 1,000" in n or "index" in n:
        return "rate/index"
    if n.startswith(("total", "number")):
        return "count"
    if "median" in n and "income" in n:
        return "dollars"
    return "other"

print("helpers ready")

helpers ready


## 2. Load all 14 year sheets

In [3]:
frames = {}
for y in YEARS:
    df = pd.read_excel(XLSX, sheet_name=str(y), header=HEADER_ROW)
    df = df.loc[:, ~df.columns.astype(str).str.startswith("Unnamed")]
    df.columns = [norm_name(c) for c in df.columns]
    frames[y] = df
    print(f"  {y}: {df.shape[0]:>3} rows x {df.shape[1]:>3} cols")

note(f"Read {len(frames)} year sheets using row {HEADER_ROW} as header "
     f"(row 0 is a column-number ruler, not data).")
note("Normalized column names so cosmetic renames between years do not "
     "create duplicate indicators.")

  2010:  56 rows x 220 cols
  2011:  56 rows x 219 cols
  2012:  56 rows x 219 cols
  2013:  56 rows x 219 cols
  2014:  56 rows x 219 cols
  2015:  56 rows x 219 cols
  2016:  56 rows x 219 cols
  2017:  56 rows x 219 cols
  2018:  56 rows x 219 cols
  2019:  56 rows x 219 cols
  2020:  56 rows x 219 cols
  2021:  56 rows x 220 cols
  2022:  56 rows x 220 cols
  2023:  56 rows x 221 cols
  NOTE: Read 14 year sheets using row 1 as header (row 0 is a column-number ruler, not data).
  NOTE: Normalized column names so cosmetic renames between years do not create duplicate indicators.


## 3. Separate the citywide benchmark

This is the answer to "all rows or just Baltimore City" in code form: the city row
comes **out** of the analysis set and is kept **aside** as a benchmark.

In [4]:
city_rows, csa_frames = [], {}
for y, df in frames.items():
    is_city = df["CSA2010"].astype(str).str.strip().eq(CITY_ROW)
    c = df[is_city].copy(); c["Year"] = y; city_rows.append(c)
    s = df[~is_city].copy(); s["Year"] = y; csa_frames[y] = s

print("CSAs per year:", sorted({len(d) for d in csa_frames.values()}))
print("Citywide rows:", sum(len(c) for c in city_rows))

note(f"Split '{CITY_ROW}' out of every sheet. It is a citywide aggregate, not "
     f"a 56th community -- leaving it in would distort every distribution, "
     f"z-score and correlation. Retained separately as a benchmark.")

CSAs per year: [55]
Citywide rows: 14
  NOTE: Split 'Baltimore City' out of every sheet. It is a citywide aggregate, not a 56th community -- leaving it in would distort every distribution, z-score and correlation. Retained separately as a benchmark.


## 4. CSA identity check

In [5]:
name_sets = {y: set(d["CSA2010"].astype(str).str.strip())
             for y, d in csa_frames.items()}
base = name_sets[2010]
mismatch = {y: s for y, s in name_sets.items() if s != base}

if mismatch:
    for y, s in mismatch.items():
        print(f"  {y} differs -- extra {sorted(s-base)} missing {sorted(base-s)}")
    note("CSA name mismatches found -- resolve before joining.")
else:
    print(f"All {len(YEARS)} sheets carry an identical set of {len(base)} CSA names.")
    note(f"CSA2010 names identical across all {len(YEARS)} sheets ({len(base)} "
         f"CSAs). CSA2010 used as the join key throughout.")

# 2020 boundary renames
if "CSA2020" in csa_frames[2023].columns:
    cmp = csa_frames[2023][["CSA2010", "CSA2020"]].dropna()
    renamed = cmp[cmp.CSA2010.astype(str).str.strip()
                  != cmp.CSA2020.astype(str).str.strip()]
    if len(renamed):
        print(f"\n{len(renamed)} CSAs renamed under 2020 boundaries:")
        for _, r in renamed.iterrows():
            print(f"   {r.CSA2010}  ->  {r.CSA2020}")
        note(f"{len(renamed)} CSAs renamed between 2010 and 2020 boundary "
             f"definitions. Joining on CSA2010; CSA2020 carried as an alternate "
             f"label. Some renames reflect real boundary redraws, so pre/post "
             f"comparisons for these seven need caution.")

All 14 sheets carry an identical set of 55 CSA names.
  NOTE: CSA2010 names identical across all 14 sheets (55 CSAs). CSA2010 used as the join key throughout.

7 CSAs renamed under 2020 boundaries:
   Claremont/Armistead  ->  Orchard Ridge/Armistead
   Greenmount East  ->  Oliver/Johnston Square
   Harford/Echodale  ->  Hamilton Hills
   Lauraville  ->  Greater Lauraville
   Medfield/Hampden/Woodberry/Remington  ->  Hampden/Remington
   Orangeville/East Highlandtown  ->  Greektown/Bayview
   Washington Village/Pigtown  ->  Pigtown/Carroll Park
  NOTE: 7 CSAs renamed between 2010 and 2020 boundary definitions. Joining on CSA2010; CSA2020 carried as an alternate label. Some renames reflect real boundary redraws, so pre/post comparisons for these seven need caution.


## 5. Reshape to tidy long format

`--` is the workbook's not-collected/suppressed token. It is coerced to null,
**never to zero** — a suppressed rate read as 0 turns a data gap into an apparently
healthy community.

In [6]:
records = []
for y, d in csa_frames.items():
    id_cols = [c for c in ("CSA2010", "CSA2020") if c in d.columns]
    val_cols = [c for c in d.columns if c not in id_cols + ["Year"]]
    m = d.melt(id_vars=id_cols, value_vars=val_cols,
               var_name="Indicator", value_name="Value")
    m["Year"] = y
    if "CSA2020" not in m.columns:
        m["CSA2020"] = np.nan
    records.append(m[["CSA2010", "CSA2020", "Year", "Indicator", "Value"]])

long_df = pd.concat(records, ignore_index=True)
long_df["CSA2010"]   = long_df["CSA2010"].astype(str).str.strip()
long_df["Indicator"] = long_df["Indicator"].map(norm_name)
long_df["ind_key"]   = long_df["Indicator"].map(key_name)

before = long_df["Value"].notna().sum()
long_df["Value"] = pd.to_numeric(long_df["Value"], errors="coerce")
after = long_df["Value"].notna().sum()

print(f"Long table: {len(long_df):,} rows")
print(f"Non-numeric placeholders coerced to null: {before - after:,}")

note("Coerced '--' and other non-numeric placeholders to null rather than "
     "zero, so data gaps are not read as healthy communities.")
long_df.head()

Long table: 167,915 rows
Non-numeric placeholders coerced to null: 5,662
  NOTE: Coerced '--' and other non-numeric placeholders to null rather than zero, so data gaps are not read as healthy communities.


,CSA2010,CSA2020,Year,Indicator,Value,ind_key
0,Allendale/Irvington/S. Hilton,NaN,2010,Total Population,16217.0,total population
1,Beechfield/Ten Hills/West Hills,NaN,2010,Total Population,12264.0,total population
2,Belair-Edison,NaN,2010,Total Population,17416.0,total population
3,Brooklyn/Curtis Bay/Hawkins Point,NaN,2010,Total Population,14243.0,total population
4,Canton,NaN,2010,Total Population,8100.0,total population


## 6. Coverage profile — which indicators are actually usable

In [7]:
cov = (long_df.groupby(["ind_key", "Year"])["Value"]
       .apply(lambda s: round(100 * s.notna().mean()))
       .unstack("Year").reindex(columns=YEARS))

cov.insert(0, "Indicator", long_df.groupby("ind_key")["Indicator"].first())
cov["years_available"] = (cov[YEARS] > 50).sum(axis=1)
cov["first_year"] = cov[YEARS].gt(50).idxmax(axis=1).where(cov.years_available > 0)
cov["last_year"]  = cov[YEARS].gt(50).apply(
    lambda r: max([y for y in YEARS if r[y]], default=np.nan), axis=1)
cov["Topic"] = cov["Indicator"].map(topic_of)
cov["Unit"]  = cov["Indicator"].map(unit_of)

always = cov[cov.years_available == len(YEARS)]
never  = cov[cov.years_available == 0]

print(f"Distinct indicators:           {len(cov)}")
print(f"Usable in all {len(YEARS)} years:        {len(always)}")
print(f"Completely empty (droppable):  {len(never)}")
print(f"Partial coverage:              {len(cov)-len(always)-len(never)}")

print("\nCoverage by topic (mean years available):")
display(cov.groupby("Topic").years_available.agg(["count", "mean"])
        .round(1).sort_values("mean"))

print("\nCompletely empty indicators -- drop these from Tableau:")
for i in never.Indicator.tolist()[:20]:
    print("   ", i)

Distinct indicators:           223
Usable in all 14 years:        39
Completely empty (droppable):  36
Partial coverage:              148

Coverage by topic (mean years available):


,count,mean
Topic,,
Civic,1,2.0
Other,54,5.1
Health,21,5.9
Safety,12,7.0
Age & Family,67,7.4
Environment,5,8.4
Income,3,9.0
Housing,15,9.6
Education,17,11.0



Completely empty indicators -- drop these from Tableau:
    Aggregate Number of Community Assets
    Air Quality Risk
    Average Distance Travelled to School (6th-8th Grade)
    Average Distance Travelled to School (9th-12th Grade)
    Average Fire and EMS Response Time
    Causes of Death (analysis of top causes)
    Causes of Death (analysis of top causes).1
    Causes of Death (analysis of top causes).2
    Causes of Death (analysis of top causes).3
    Energy Use
    Fetal Mortality Rate
    Natural gas utilization
    Number of Bike Racks per 1,000 Residents
    Number of Children Enrolled in Head Start Programs
    Number of Downspout Disconnections per 1,000 Residences
    Number of Persons calling UWCM 211 for Health-Related Services
    Number of Photovoltaic/Solar Permits
    Number of Public Trashcans
    Percent of Area Covered by Impervious Surface
    Percent of CSA that is Either Low or Moderate income (by Census tract)


## 7. Temporal alignment — the critical finding

Health outcomes stop years before the socioeconomic series ends. This determines
which year the correlation analysis can honestly use.

In [8]:
health = cov[(cov.Topic == "Health") & (cov.years_available > 0)]
other  = cov[(cov.Topic != "Health") & (cov.years_available > 0)]
print("Latest year with ANY health indicator:     ", int(health.last_year.max()))
print("Latest year with socioeconomic indicators: ", int(other.last_year.max()))

overlap = pd.DataFrame([{
    "Year": y,
    "health_indicators": int((cov.loc[cov.Topic == "Health", y] > 50).sum()),
    "ses_indicators": int((cov.loc[cov.Topic.isin(
        ["Income", "Access", "Education", "Employment"]), y] > 50).sum()),
} for y in YEARS])
display(overlap)

# Choose the cross-section by RICHEST health coverage, not the latest year
# containing any health field. 2019 retains only birth outcomes; life
# expectancy and mortality -- the headline outcomes -- end in 2018.
best = overlap[overlap.health_indicators == overlap.health_indicators.max()]
ALIGN_YEAR, CURRENT_YEAR = int(best.Year.max()), 2023

print(f"\n>>> Richest health coverage: {int(best.health_indicators.max())} "
      f"indicators. Latest such year = {ALIGN_YEAR}")
print(f">>> Current-conditions year  = {CURRENT_YEAR}")

note(f"Health outcomes end by 2019 while socioeconomic indicators run to 2023; "
     f"no recent year contains both. Correlation analysis uses {ALIGN_YEAR} as "
     f"an internally consistent cross-section, and {CURRENT_YEAR} separately "
     f"for current conditions. Combining the two vintages into a single score "
     f"would be indefensible unless explicitly disclosed.")
note("No emergency-department or health-insurance indicators exist anywhere in "
     "this workbook; any argument resting on them must be sourced elsewhere.")

Latest year with ANY health indicator:      2019
Latest year with socioeconomic indicators:  2023


,Year,health_indicators,ses_indicators
0,2010,6,26
1,2011,14,37
2,2012,14,37
3,2013,14,37
4,2014,14,36
5,2015,14,36
6,2016,14,37
7,2017,14,37
8,2018,14,21
9,2019,6,37



>>> Richest health coverage: 14 indicators. Latest such year = 2018
>>> Current-conditions year  = 2023
  NOTE: Health outcomes end by 2019 while socioeconomic indicators run to 2023; no recent year contains both. Correlation analysis uses 2018 as an internally consistent cross-section, and 2023 separately for current conditions. Combining the two vintages into a single score would be indefensible unless explicitly disclosed.
  NOTE: No emergency-department or health-insurance indicators exist anywhere in this workbook; any argument resting on them must be sourced elsewhere.


## 8. Outliers and range checks

In [9]:
flags = []
for (ind, y), g in long_df.dropna(subset=["Value"]).groupby(["ind_key", "Year"]):
    v = g["Value"]
    if len(v) < 10:
        continue
    q1, q3 = v.quantile([.25, .75]); iqr = q3 - q1
    if iqr == 0:
        continue
    lo, hi = q1 - 3*iqr, q3 + 3*iqr
    for _, r in g[(v < lo) | (v > hi)].iterrows():
        flags.append({"Indicator": r["Indicator"], "Year": y,
                      "CSA2010": r["CSA2010"], "Value": r["Value"],
                      "q1": round(q1, 2), "q3": round(q3, 2),
                      "reason": "beyond 3xIQR"})

# Percent indicators outside 0-100. Exclude change/growth measures --
# "Percent Population Change from last Census" is legitimately negative in a
# shrinking city and is not a scale error.
pct_keys = set(cov[(cov.Unit == "percent") & ~cov.Indicator.str.contains(
    "change|growth|difference", case=False)].index)
pr = long_df[long_df.ind_key.isin(pct_keys)].dropna(subset=["Value"])
bad_pct = pr[(pr.Value < 0) | (pr.Value > 100)]
for _, r in bad_pct.iterrows():
    flags.append({"Indicator": r["Indicator"], "Year": r["Year"],
                  "CSA2010": r["CSA2010"], "Value": r["Value"],
                  "q1": np.nan, "q3": np.nan,
                  "reason": "percent outside 0-100"})

outliers = pd.DataFrame(flags)
print(f"Statistical outliers (>3x IQR): {len(flags)-len(bad_pct):,}")
print(f"Percent values outside 0-100:   {len(bad_pct):,}")

if len(outliers):
    print("\nCSAs most often flagged:")
    print(outliers.CSA2010.value_counts().head(6).to_string())
    note(f"{len(outliers):,} outlier values flagged for review, not removal. "
         f"In this dataset extremes are usually real -- Downtown/Seton Hill has "
         f"a small residential base and large daytime population, and "
         f"institutional CSAs legitimately differ -- rather than errors.")

Statistical outliers (>3x IQR): 1,366
Percent values outside 0-100:   5

CSAs most often flagged:
CSA2010
Downtown/Seton Hill                358
Midtown                             98
Oldtown/Middle East                 96
Inner Harbor/Federal Hill           67
Greater Roland Park/Poplar Hill     56
South Baltimore                     55
  NOTE: 1,371 outlier values flagged for review, not removal. In this dataset extremes are usually real -- Downtown/Seton Hill has a small residential base and large daytime population, and institutional CSAs legitimately differ -- rather than errors.


## 9. Correlation scan at the aligned year

Report these as **ecological correlations across 55 areas, not individual-level
effects**, and say so when presenting.

In [10]:
wide_align = (long_df[long_df.Year == ALIGN_YEAR]
              .pivot_table(index="CSA2010", columns="Indicator",
                           values="Value", aggfunc="first"))
usable = wide_align.loc[:, wide_align.notna().sum() >= 40]
print(f"Indicators with >=40/55 CSAs populated in {ALIGN_YEAR}: {usable.shape[1]}")

LE = [c for c in usable.columns if "life expectancy" in c.lower()]
if LE:
    le = LE[0]
    print(f"\nLife expectancy {ALIGN_YEAR}:  min {usable[le].min():.1f}   "
          f"max {usable[le].max():.1f}   "
          f"GAP {usable[le].max()-usable[le].min():.1f} years")

    print("\nLowest 5:")
    display(usable[le].nsmallest(5).round(1).to_frame())
    print("Highest 5:")
    display(usable[le].nlargest(5).round(1).to_frame())

    cors = usable.corrwith(usable[le]).drop(le).dropna().sort_values()
    print("\nMost NEGATIVE correlates (worse outcome as this rises):")
    display(cors.head(12).round(3).to_frame("r"))
    print("Most POSITIVE correlates:")
    display(cors.tail(12).round(3).to_frame("r"))
else:
    print("Life expectancy unavailable in the aligned year.")

Indicators with >=40/55 CSAs populated in 2018: 122

Life expectancy 2018:  min 63.2   max 84.7   GAP 21.6 years

Lowest 5:


,Life Expectancy
CSA2010,
Downtown/Seton Hill,63.2
Southwest Baltimore,66.4
Pimlico/Arlington/Hilltop,67.1
Poppleton/The Terraces/Hollins Market,67.2
Clifton-Berea,67.4


Highest 5:


,Life Expectancy
CSA2010,
Cross-Country/Cheswolde,84.7
Greater Roland Park/Poplar Hill,82.7
North Baltimore/Guilford/Homeland,82.6
Inner Harbor/Federal Hill,80.8
Canton,80.8



Most NEGATIVE correlates (worse outcome as this rises):


,r
Indicator,
Mortality by Age (45-64 Years old),-0.913
Percent of Female-Headed Households with Children Under 18,-0.827
Mortality by Age (25-44 Years old),-0.812
Percentage of Residential Properties with Housing Violations (Excluding Vacants),-0.797
Percent of Families Receiving TANF,-0.791
"Number of Shootings per 1,000 Residents",-0.770
Percent of Households with No Vehicles Available,-0.759
Mortality by Age (65-84 Years old),-0.750
"Percent of Households Earning Less than $25,000",-0.743


Most POSITIVE correlates:


,r
Indicator,
Percent of Births Delivered at Term (37-42 Weeks),0.610
Percent Population 16-64 Employed,0.655
"Number of Homestead Tax Credits per 1,000 Residential Units",0.685
Percentage of Housing Units that are Owner-Occupied,0.685
Percent of Babies Born with a Satisfactory Birth Weight,0.694
Percent of Population that Drove Alone to Work,0.697
Percent Population (25 Years and over) with a Bachelor's Degree or Above,0.722
Percent of Residents - White/Caucasian (Non-Hispanic),0.738
Median Price of Homes Sold,0.784


## 10. Export for Tableau

In [11]:
long_df["Topic"] = long_df["Indicator"].map(topic_of)
long_df["Unit"]  = long_df["Indicator"].map(unit_of)

# Drop all-empty indicators so they don't clutter Tableau's field list
tidy = long_df[~long_df.ind_key.isin(set(never.index))].copy()
print(f"Dropped {len(never)} all-empty indicators from the export")


# Pre-compute z-scores and percentile ranks so Tableau needs no table calcs
tz = tidy.dropna(subset=["Value"]).copy()
g = tz.groupby(["ind_key", "Year"])["Value"]
tz["z_score"]   = ((tz.Value - g.transform("mean"))
                   / g.transform("std").replace(0, np.nan)).round(4)
tz["pct_rank"]  = g.rank(pct=True).round(4)

# Cross-sections
for yr in (ALIGN_YEAR, CURRENT_YEAR):
    w = (tidy[tidy.Year == yr].pivot_table(index="CSA2010", columns="Indicator",
                                           values="Value", aggfunc="first"))
    w = w.loc[:, w.notna().sum() >= 40]
    print(f"  csa_wide_{yr}.csv: {w.shape[0]} CSAs x {w.shape[1]} indicators")

# Citywide benchmark, for reference lines
city = pd.concat(city_rows, ignore_index=True)
city.columns = [norm_name(c) for c in city.columns]
cid = [c for c in ("CSA2010", "CSA2020") if c in city.columns]
cm = city.melt(id_vars=cid + ["Year"], var_name="Indicator", value_name="CityValue")
cm["Indicator"] = cm["Indicator"].map(norm_name)
cm["CityValue"] = pd.to_numeric(cm["CityValue"], errors="coerce")

print()
for f in sorted(os.listdir(OUT)):
    print(f"  {f:38s} {os.path.getsize(os.path.join(OUT, f)):>10,} bytes")

Dropped 36 all-empty indicators from the export
  csa_wide_2018.csv: 55 CSAs x 122 indicators
  csa_wide_2023.csv: 55 CSAs x 113 indicators

  citywide_benchmark.csv                    124,447 bytes
  cleaning_log.md                             1,946 bytes
  correlations_life_expectancy.csv            7,214 bytes
  csa2010_to_csa2020_crosswalk.csv              312 bytes
  csa_indicator_year_long.csv            15,720,461 bytes
  csa_indicator_year_scored.csv          11,069,817 bytes
  csa_wide_2018.csv                         107,455 bytes
  csa_wide_2023.csv                          87,774 bytes
  indicator_coverage.csv                     33,617 bytes
  indicator_dictionary.csv                   17,836 bytes
  outlier_flags.csv                         156,436 bytes


## 11. Single Excel workbook for Tableau

Tableau reads each worksheet in an `.xlsx` as its own table, so one file replaces the
loose CSVs. Three sheets:

| Sheet | Grain | Answers | Score? |
|---|---|---|---|
| `2023_Analysis` | one row per CSA | where is need highest **now** | yes, appended at the right |
| `2018_Analysis` | one row per CSA | what **drives** health outcomes | no |
| `Historical_Trends` | CSA x indicator x year | how things have moved | no |

2018 is the last cross-section holding life expectancy and mortality; 2023 holds no
health indicators at all. The score is deliberately 2023-only -- a percentile rank is
relative to the other CSAs in its own year, so comparing scores across years would show
reshuffling rather than progress.


In [ ]:
# Same content as the CSV exports above, packaged as one workbook -- Tableau reads
# each worksheet as its own table.
#   <CURRENT_YEAR>_Analysis : every indicator, plus the computed score block at the far right
#   <ALIGN_YEAR>_Analysis   : every indicator INCLUDING the 14 health outcomes, values only
#   Historical_Trends       : all indicators, all years, long format
# Only the current year carries a score: see the note at the end of this cell. There is no
# separate current-year long sheet -- it would be Historical_Trends filtered to one year,
# identical column for column, so filter in Tableau instead.
sheet_cols = ["CSA", "CSA2020", "Year", "Category", "Indicator", "Unit", "Value"]
# CityValue is appended to the long sheet from the held-out benchmark row.

src = (tidy.rename(columns={"CSA2010": "CSA", "Topic": "Category"})
       .dropna(subset=["Value"]))

# --- the 9 "higher is worse" indicators behind the Service Need Score
NEED = [
    "Percent of Family Households Living Below the Poverty Line",
    "Percent of Children Living Below the Poverty Line",
    "Unemployment Rate",
    "Percent Population (25 Years and over) With Less Than a High School Diploma or GED",
    "Percent of Households with No Internet at Home",
    "Percent of Households with No Vehicles Available",
    "Percentage of Residential Properties that are Vacant and Abandoned",
    "Rate of Evictions per 1,000 Residents",
    "Number of Overdose Calls for Service per 1,000 Residents",
]
SHORT = ["Family Households Below Poverty", "Children Below Poverty", "Unemployment Rate",
         "Adults Without HS Diploma", "Households With No Internet",
         "Households With No Vehicle", "Vacant & Abandoned Residential",
         "Evictions per 1,000 Residents", "Overdose Calls per 1,000 Residents"]

def band(v):
    return ("Highest relative need"  if v >= 75 else
            "Higher relative need"   if v >= 50 else
            "Moderate relative need" if v >= 25 else "Lower relative need")

def wide(year):
    """One row per CSA, one column per indicator populated for >=40 of the 55 CSAs."""
    w = (src[src.Year == year]
         .pivot_table(index="CSA", columns="Indicator", values="Value", aggfunc="first"))
    return w.loc[:, w.notna().sum() >= 40]

# --- Sheet 1: current year, all indicators, then the computed block at the end
w_cur = wide(CURRENT_YEAR)
missing = [c for c in NEED if c not in w_cur.columns]
if missing:
    raise KeyError(f"Score indicators absent from {CURRENT_YEAR}: {missing}")

ranks = w_cur[NEED].rank(method="average") / len(w_cur)   # unrounded; the score averages these
wide_cur = w_cur.copy()
for c, s in zip(NEED, SHORT):
    wide_cur[f"{s} (Percentile)"] = ranks[c].round(6)
wide_cur["Service Need Score"] = (ranks.mean(axis=1) * 100).round(5)
wide_cur["Interpretation"]     = wide_cur["Service Need Score"].map(band)
wide_cur = wide_cur.sort_values("Service Need Score", ascending=False).reset_index()

# --- Sheet 2: aligned year -- the only cross-section with health outcomes. Values only.
w_align   = wide(ALIGN_YEAR)
wide_align = w_align.sort_values(w_align.columns[0]).reset_index()
n_health  = (src[src.Indicator.isin(w_align.columns)]
             .groupby("Indicator").Category.first().eq("Health").sum())
n_score   = sum(c in w_align.columns for c in NEED)

# --- Sheet 3: long format, every year, with the citywide row alongside each value.
# "Baltimore City" is held out of the 55 CSAs (section 3) but carried here as CityValue
# so Tableau can draw a reference line without a second data source.
bench = (cm.dropna(subset=["CityValue"])[["Year", "Indicator", "CityValue"]]
         .drop_duplicates(["Year", "Indicator"]))
hist = (src[sheet_cols]
        .merge(bench, on=["Year", "Indicator"], how="left")
        .sort_values(["Indicator", "Year", "CSA"]).reset_index(drop=True))
matched = hist["CityValue"].notna().mean() * 100

note(f"Packaged the section 10 exports as one workbook. The Service Need Score is an "
     f"unweighted mean of {len(NEED)} percentile ranks -- equal weighting is a choice, not a "
     f"finding, and child/family poverty are near-duplicates, so present it as a screening "
     f"tool for prioritisation rather than a measurement of need.")
note(f"Only the {CURRENT_YEAR} sheet carries a score, for two reasons. Just {n_score} of the "
     f"{len(NEED)} components exist in {ALIGN_YEAR} (evictions are not collected then), so the "
     f"two scores would not measure the same thing. More fundamentally the ranks are relative "
     f"to the other {len(w_cur)-1} CSAs within a single year: if every CSA improved equally "
     f"the score would not move, so a {ALIGN_YEAR}-to-{CURRENT_YEAR} score comparison shows "
     f"reshuffling, never progress. Use raw values from Historical_Trends for change over time.")
note(f"{ALIGN_YEAR}_Analysis exists because it is the last cross-section holding health "
     f"outcomes: {n_health} health indicators including life expectancy and mortality by age, "
     f"none of which are collected after 2019. It is the source for health-vs-socioeconomic "
     f"scatterplots, which need indicators as separate columns.")


XLSX_OUT = f"{OUT}/Baltimore_Vital_Signs_Tableau.xlsx"
sheets = [(f"{CURRENT_YEAR}_Analysis", wide_cur), (f"{ALIGN_YEAR}_Analysis", wide_align),
          ("Historical_Trends", hist)]
with pd.ExcelWriter(XLSX_OUT, engine="openpyxl") as xw:
    for name, df in sheets:
        df.to_excel(xw, sheet_name=name, index=False)
    from openpyxl.styles import Font, Alignment
    for name, df in sheets:
        ws = xw.sheets[name]
        ws.freeze_panes = "B2"
        ws.auto_filter.ref = ws.dimensions
        for j, col in enumerate(df.columns, start=1):
            cell = ws.cell(row=1, column=j)
            cell.font = Font(bold=True)
            cell.alignment = Alignment(wrap_text=True, vertical="top")
            ws.column_dimensions[cell.column_letter].width = min(max(len(str(col)) + 2, 12), 42)

print(XLSX_OUT)
for name, df in sheets:
    print(f"  {name:<18} {df.shape[0]:>7,} rows x {df.shape[1]:>3} cols")
print(f"\n{CURRENT_YEAR}_Analysis: {w_cur.shape[1]} indicators + {len(NEED)} percentiles "
      f"+ score + band")
print(f"{ALIGN_YEAR}_Analysis: {w_align.shape[1]} indicators, {n_health} of them health, "
      f"no score block")
print(f"shared between the two wide sheets: "
      f"{len(set(w_cur.columns) & set(w_align.columns))} indicators")
print(f"Historical_Trends rows with a citywide benchmark: {matched:.0f}%")
display(wide_cur[["CSA", "Service Need Score", "Interpretation"]].head(10))


## 12. What tracks health outcomes, and how that changed

Correlations for **every year each outcome exists**, so you can chart `r` over time
rather than read it once. Two sheets:

| Sheet | Grain | Chart it as |
|---|---|---|
| `Correlations` | outcome x year x indicator | line of `r` over `Year`; or bar at one year |
| `Correlation_Change` | outcome x indicator | bar of `r_change`, first year vs last |

**The 2023 constraint.** No true health outcome survives past 2019, so the correlates of
life expectancy cannot be extended to 2023 -- there is no 2023 life expectancy to
correlate against. Narcotics calls (2011-2023) and overdose calls (2018-2023) carry the
trend into the present instead. They are service-demand proxies, not health status.

Health indicators are never predictors, and collinear predictors are pruned once at each
outcome's latest year so trend lines stay comparable year to year.


In [ ]:
# Correlations computed for EVERY year each outcome exists, so the relationship can be
# tracked over time rather than read once at ALIGN_YEAR.
#
# The hard constraint: no true health outcome survives past 2019, so "how did the
# correlates of life expectancy change by 2023" is unanswerable -- there is no 2023 life
# expectancy. Two health-adjacent outcomes do reach 2023 and carry the trend instead.
OUTCOMES = {
    # outcome: (higher value of the outcome is good?, why it is here)
    "Life Expectancy":                                    (True,  "headline, 2011-2018"),
    "Infant Mortality Rate":                              (False, "headline, 2011-2018"),
    "Teen Pregnancy Rate per 1,000 Females (aged 15-19)": (False, "2010-2019"),
    "Percent of Babies Born with a Satisfactory Birth Weight": (True, "2010-2019"),
    "Number of Narcotics Calls for Service per 1,000 Residents": (False, "2011-2023, spans both eras"),
    "Number of Overdose Calls for Service per 1,000 Residents":  (False, "2018-2023, reaches today"),
}
COLLIN_MAX, MIN_N = 0.85, 40

cat = src.groupby("Indicator").Category.first()
wide_by_year = {y: wide(y) for y in YEARS}

rows = []
for outcome, (higher_is_good, _why) in OUTCOMES.items():
    years = [y for y in YEARS
             if outcome in wide_by_year[y].columns
             and wide_by_year[y][outcome].notna().sum() >= MIN_N]
    if not years:
        note(f"Correlation scan: '{outcome}' never reaches {MIN_N} CSAs, skipped.")
        continue

    # Prune collinear predictors ONCE, at the outcome's most recent year, and reuse that
    # decision for every year. Re-pruning per year would let indicators drop in and out
    # and turn a trend line into an artefact of the pruning.
    ref = wide_by_year[max(years)]
    ref_preds = [c for c in ref.columns if c != outcome and cat.get(c) != "Health"]
    ref_r = ref[ref_preds].corrwith(ref[outcome]).dropna()
    kept = []
    for c in ref_r.abs().sort_values(ascending=False).index:
        if all(abs(ref[[c, k]].corr().iloc[0, 1]) <= COLLIN_MAX for k in kept):
            kept.append(c)
    kept = set(kept)

    for y in years:
        w = wide_by_year[y]
        preds = [c for c in w.columns if c != outcome and cat.get(c) != "Health"]
        r = w[preds].corrwith(w[outcome]).dropna()
        for c in r.index:
            rows.append({
                "Outcome": outcome, "Year": y, "Indicator": c,
                "Category": cat.get(c, "Other"),
                "r": round(float(r[c]), 4), "abs_r": round(abs(float(r[c])), 4),
                "n": int(w[[c, outcome]].dropna().shape[0]),
                "Keep": bool(c in kept),
                "Relationship": ("higher goes with a better outcome"
                                 if (r[c] > 0) == higher_is_good
                                 else "higher goes with a worse outcome"),
            })

corr_tbl = pd.DataFrame(rows)
latest = corr_tbl.groupby("Outcome").Year.transform("max")
corr_tbl["IsLatestYear"] = corr_tbl.Year.eq(latest)
corr_tbl = corr_tbl.sort_values(["Outcome", "Year", "abs_r"],
                                ascending=[True, True, False]).reset_index(drop=True)

# How far has each relationship moved between the outcome's first and last year?
first_last = []
for (o, i), g in corr_tbl[corr_tbl.Keep].groupby(["Outcome", "Indicator"]):
    g = g.sort_values("Year")
    if len(g) < 2:
        continue
    first_last.append({"Outcome": o, "Indicator": i, "Category": g.Category.iloc[0],
                       "first_year": int(g.Year.iloc[0]), "last_year": int(g.Year.iloc[-1]),
                       "r_first": g.r.iloc[0], "r_last": g.r.iloc[-1],
                       "r_change": round(g.r.iloc[-1] - g.r.iloc[0], 4)})
corr_change = (pd.DataFrame(first_last)
               .sort_values("r_change", key=abs, ascending=False).reset_index(drop=True))

note(f"Correlations recomputed for every year each outcome exists, not just {ALIGN_YEAR}, "
     f"so relationships can be tracked over time. Collinearity pruning is decided once at "
     f"each outcome's most recent year and reused, otherwise indicators drop in and out and "
     f"the trend line becomes an artefact of the pruning.")
note("No true health outcome exists after 2019, so the correlates of life expectancy cannot "
     "be extended to 2023. Narcotics calls (2011-2023) and overdose calls (2018-2023) are "
     "carried as health-adjacent outcomes that do reach the present; they are service-demand "
     "proxies, not health status, and should be labelled as such.")

with pd.ExcelWriter(XLSX_OUT, engine="openpyxl", mode="a", if_sheet_exists="replace") as xw:
    corr_tbl.to_excel(xw, sheet_name="Correlations", index=False)
    corr_change.to_excel(xw, sheet_name="Correlation_Change", index=False)
    from openpyxl.styles import Font, Alignment
    for nm, df in [("Correlations", corr_tbl), ("Correlation_Change", corr_change)]:
        ws = xw.sheets[nm]; ws.freeze_panes = "A2"; ws.auto_filter.ref = ws.dimensions
        for j, col in enumerate(df.columns, start=1):
            c0 = ws.cell(row=1, column=j)
            c0.font = Font(bold=True); c0.alignment = Alignment(wrap_text=True, vertical="top")
            ws.column_dimensions[c0.column_letter].width = min(max(len(str(col)) + 2, 12), 46)

print(f"Correlations      : {len(corr_tbl):,} rows, {corr_tbl.Outcome.nunique()} outcomes")
print(corr_tbl.groupby('Outcome').Year.agg(['min','max','nunique'])
      .rename(columns={'nunique':'years'}).to_string())
print(f"\nCorrelation_Change: {len(corr_change):,} rows")
print("\nBiggest shifts in relationship (pruned predictors only):")
for _, x in corr_change.head(8).iterrows():
    print(f"  {x.r_first:+.2f} -> {x.r_last:+.2f} ({x.r_change:+.2f})  "
          f"{x.Outcome[:28]:28s} vs {x.Indicator[:40]}")

## 13. Grouping the indicators

Three groupings, because they answer different questions:

| Axis | What it captures | Use it for |
|---|---|---|
| `Category` | what the indicator is *about* | filtering and colour in Tableau |
| `Unit` | percent / rate / count | counts are not comparable across CSAs -- keep them off choropleths |
| `Construct` | what the indicator *measures* | picking one per construct so a composite does not double-count |

Constructs are derived from the data rather than assigned by hand: indicators that
correlate above 0.90 at the aligned year are treated as one underlying thing, and the
best-covered member names the group.


In [ ]:
# Grouping the indicators, on three axes that answer different questions.
#   Category  - what the indicator is ABOUT. Use it to filter and colour.
#   Unit      - percent / rate / count. Counts are not comparable across CSAs of
#               different size; only percents and rates belong on a choropleth.
#   Construct - what the indicator MEASURES. Derived from the data: indicators
#               correlating above CLUSTER_AT are the same underlying thing wearing
#               different names. Pick ONE per construct or you weight it twice.
CLUSTER_AT = 0.90

wa = wide(ALIGN_YEAR)
cols = list(wa.columns)
cmat = wa.corr().abs()   # NOT cm -- that name holds the citywide melt from section 10

# connected components over the "correlates above CLUSTER_AT" graph
adj = {c: {d for d in cols if d != c and cmat.loc[c, d] > CLUSTER_AT} for c in cols}
seen, clusters = set(), []
for c in cols:
    if c in seen:
        continue
    stack, comp = [c], []
    while stack:
        x = stack.pop()
        if x in seen:
            continue
        seen.add(x); comp.append(x)
        stack.extend(adj[x] - seen)
    clusters.append(sorted(comp))

cov_by_ind = src.groupby("Indicator").Value.count()
label_of, id_of, size_of = {}, {}, {}
for k, comp in enumerate(sorted(clusters, key=len, reverse=True), start=1):
    lead = max(comp, key=lambda c: cov_by_ind.get(c, 0))   # best-covered member names it
    for c in comp:
        label_of[c], id_of[c], size_of[c] = lead, k, len(comp)

dic = (src.groupby("Indicator")
       .agg(Category=("Category", "first"), Unit=("Unit", "first"),
            first_year=("Year", "min"), last_year=("Year", "max"),
            non_null_values=("Value", "count"))
       .reset_index())
dic["Construct"]     = dic.Indicator.map(label_of)
dic["ConstructID"]   = dic.Indicator.map(id_of)
dic["ConstructSize"] = dic.Indicator.map(size_of).fillna(1).astype("Int64")
dic["IsConstructLead"] = dic.Indicator.eq(dic.Construct)
dic["InAlignYear"]   = dic.Indicator.isin(cols)
dic = dic.sort_values(["Category", "Indicator"]).reset_index(drop=True)

multi = dic[dic.ConstructSize > 1]
note(f"Grouped indicators on three axes: Category (what it is about), Unit (percent/rate/"
     f"count -- counts must not go on a choropleth), and Construct. Constructs are derived "
     f"empirically at {ALIGN_YEAR}: indicators correlating above {CLUSTER_AT} are the same "
     f"thing under different names. {multi.ConstructID.nunique()} constructs contain more "
     f"than one indicator, covering {len(multi)} indicators -- pick the lead of each, or a "
     f"composite weights that construct multiple times.")

with pd.ExcelWriter(XLSX_OUT, engine="openpyxl", mode="a", if_sheet_exists="replace") as xw:
    dic.to_excel(xw, sheet_name="Indicator_Dictionary", index=False)
    from openpyxl.styles import Font, Alignment
    ws = xw.sheets["Indicator_Dictionary"]
    ws.freeze_panes = "B2"; ws.auto_filter.ref = ws.dimensions
    for j, col in enumerate(dic.columns, start=1):
        c0 = ws.cell(row=1, column=j)
        c0.font = Font(bold=True); c0.alignment = Alignment(wrap_text=True, vertical="top")
        ws.column_dimensions[c0.column_letter].width = min(max(len(str(col)) + 2, 12), 46)

print(f"Indicator_Dictionary: {len(dic)} indicators")
print(f"\nBy Category:\n{dic.Category.value_counts().to_string()}")
print(f"\nBy Unit:\n{dic.Unit.value_counts().to_string()}")
print(f"\nMulti-indicator constructs (pick one member of each): "
      f"{multi.ConstructID.nunique()} covering {len(multi)} indicators")
for k, g in multi.groupby("ConstructID"):
    if len(g) < 3:
        continue
    print(f"\n  construct {k} -- lead: {g.Construct.iloc[0][:56]}")
    for i in g.Indicator:
        if i != g.Construct.iloc[0]:
            print(f"      also: {i[:62]}")

## 14. Answering the brief: who needs navigation, advocacy, literacy and caregiver support

The `Service Need Score` in section 11 is a **general deprivation index** and does not
answer this question. These services have distinct need signatures, so each is scored on
its own dimension:

| Dimension | Proxied by |
|---|---|
| Health literacy | adults without a HS diploma |
| Navigation barriers | no vehicle (reaching appointments), no internet (portals, telehealth) |
| Caregiver load | population 65+, female-headed households with children |
| Complex care need | overdose and narcotics calls per 1,000 |

The dimensions are kept visible rather than collapsed, because they disagree: caregiver
load correlates only ~0.39 with health-literacy need. A single ranking would mis-target
at least one service line.

**Known gaps:** no health-insurance indicator anywhere in the workbook (the column exists
but is empty for all 14 years), no disability measure, no English-proficiency measure.


In [ ]:
# The brief's question: which communities most need healthcare navigation, patient
# advocacy, health literacy or caregiver support?
#
# The Service Need Score does NOT answer this -- it is a general deprivation index. These
# services have distinct need signatures, so each is scored on its own dimension and the
# dimensions are kept visible rather than collapsed into one number.
DIMENSIONS = {
    "Health literacy": [
        "Percent Population (25 Years and over) With Less Than a High School Diploma or GED"],
    "Navigation barriers": [
        "Percent of Households with No Vehicles Available",      # getting to an appointment
        "Percent of Households with No Internet at Home"],       # portals, telehealth, booking
    "Caregiver load": [
        "Percent of Population 65 Years and over",
        "Percent of Female-Headed Households with Children Under 18"],
    "Complex care need": [
        "Number of Overdose Calls for Service per 1,000 Residents",
        "Number of Narcotics Calls for Service per 1,000 Residents"],
}
CONTEXT = {"Percent of Residents - Hispanic": "Pct Hispanic",
           "Median Household Income": "Median Household Income"}

wc = wide(CURRENT_YEAR)
missing = [c for cols in DIMENSIONS.values() for c in cols if c not in wc.columns]
if missing:
    raise KeyError(f"Service-need indicators absent from {CURRENT_YEAR}: {missing}")

prof = pd.DataFrame(index=wc.index)
for dim, cols in DIMENSIONS.items():
    # percentile within each indicator, then average -> the dimension is on a 0-100 scale
    prof[dim] = (wc[cols].rank(pct=True).mean(axis=1) * 100).round(1)

# Equal weight ACROSS DIMENSIONS, not across indicators: the four are conceptually
# distinct, whereas averaging the 7 raw indicators would weight navigation and complex
# care (2 indicators each) above health literacy (1).
prof["Composite"] = prof[list(DIMENSIONS)].mean(axis=1).round(1)
prof["Primary need"] = prof[list(DIMENSIONS)].idxmax(axis=1)
prof["Tier"] = pd.cut(prof.Composite, [-0.1, 40, 55, 70, 100],
                      labels=["Lower", "Moderate", "High", "Highest"]).astype(str)

wal = wide(ALIGN_YEAR)
if "Life Expectancy" in wal.columns:
    prof[f"Life expectancy {ALIGN_YEAR}"] = wal["Life Expectancy"].round(1)
    city_le = cm[(cm.Year == ALIGN_YEAR) &
                 (cm.Indicator == "Life Expectancy")].CityValue
    if len(city_le):
        prof[f"Years below city {ALIGN_YEAR}"] = (
            wal["Life Expectancy"] - float(city_le.iloc[0])).round(1)
for col, label in CONTEXT.items():
    if col in wc.columns:
        prof[label] = wc[col].round(1)

# --- Persistence: how many years this CSA ranked among the 10 worst for life
# expectancy. Chronic failure is a different proposition from one bad year, and this
# is what separates a first placement from a second-wave one.
WORST_N = 10
le_long = src[src.Indicator == "Life Expectancy"].dropna(subset=["Value"])
le_years = sorted(y for y in le_long.Year.unique()
                  if le_long[le_long.Year == y].CSA.nunique() >= 40)
pcount = pd.Series(0, index=prof.index, dtype="int64")
for y in le_years:
    # lowest Value first, CSA name as a deterministic tiebreak. 2017 has two CSAs
    # tied on the 10th-lowest value, so without a stable tiebreak the result would
    # depend on row order.
    worst = (le_long[le_long.Year == y]
             .sort_values(["Value", "CSA"], kind="mergesort")
             .CSA.head(WORST_N))
    pcount[pcount.index.intersection(worst)] += 1
prof["Persistence_Count"] = pcount

unmatched = sorted(set(prof.index) - set(le_long.CSA))
if unmatched:
    note(f"Persistence_Count: {len(unmatched)} CSAs have no life-expectancy data and "
         f"scored 0 by default -- {unmatched}")

profile = (prof.sort_values("Composite", ascending=False)
           .reset_index().rename(columns={"CSA2010": "CSA"}))

note("Added a service-need profile answering the brief directly. Each service line is "
     "scored separately because they have different geographies: caregiver load correlates "
     "only 0.39 with health literacy need, so a single ranking would mis-target at least "
     "one of them.")
note("Service-need gaps: the workbook has NO health-insurance indicator (the column exists "
     "but is empty in all 14 years), no disability measure and no English-proficiency "
     "measure. Uninsurance is arguably the central navigation-need variable and cannot be "
     "sourced here. Pct Hispanic is carried as a rough language-access proxy only.")

with pd.ExcelWriter(XLSX_OUT, engine="openpyxl", mode="a", if_sheet_exists="replace") as xw:
    profile.to_excel(xw, sheet_name="Service_Need_Profile", index=False)
    from openpyxl.styles import Font, Alignment
    ws = xw.sheets["Service_Need_Profile"]
    ws.freeze_panes = "B2"; ws.auto_filter.ref = ws.dimensions
    for j, col in enumerate(profile.columns, start=1):
        c0 = ws.cell(row=1, column=j)
        c0.font = Font(bold=True); c0.alignment = Alignment(wrap_text=True, vertical="top")
        ws.column_dimensions[c0.column_letter].width = min(max(len(str(col)) + 2, 12), 34)

print(f"Service_Need_Profile: {len(profile)} CSAs x {profile.shape[1]} cols")
print(f"Persistence_Count over {len(le_years)} years ({le_years[0]}-{le_years[-1]}), "
      f"worst {WORST_N} each year; total flagged = "
      f"{int(profile.Persistence_Count.sum())} (expected {len(le_years)*WORST_N})\n")
print(profile.head(10)[["CSA"] + list(DIMENSIONS) +
                       ["Composite", "Primary need"]].to_string(index=False))
print("\nHighest on each dimension individually:")
for d in DIMENSIONS:
    t = profile.nlargest(4, d)
    print(f"  {d:20s} " + ", ".join(f"{r.CSA.split('/')[0][:20]} ({r[d]:.0f})"
                                    for _, r in t.iterrows()))
print("\nHow much do the dimensions agree? (rank correlation)")
print(profile[list(DIMENSIONS)].rank().corr().round(2).to_string())

---

## What to do next

### The output

Everything lands in a single workbook, `tableau/Baltimore_Vital_Signs_Tableau.xlsx`:

| Sheet | Use |
|---|---|
| `2023_Analysis` | current conditions map; Service Need Score and bands at the far right |
| `2018_Analysis` | health-vs-socioeconomic scatterplots — the only year holding both |
| `Historical_Trends` | trends over time, with the citywide figure alongside each value as `CityValue` |
| `Correlations` | what tracks each outcome, by year — line chart of `r` over `Year` |
| `Correlation_Change` | how each relationship shifted, first year vs last |
| `Indicator_Dictionary` | Category / Unit / Construct — pick one indicator per construct |
| `Service_Need_Profile` | the brief's question: who needs navigation, literacy, caregiver support |

Join the `CSA` field to a Community Statistical Area boundary file for the map.

### Still worth doing

- **Spatial autocorrelation** once boundaries are joined, to show deprivation is clustered
  rather than scattered. Needs `geopandas`.
- **Rate reliability for small-base CSAs.** Downtown/Seton Hill has a tiny residential
  population against a large daytime one, and is the most-flagged outlier in section 8.
  Flag low-denominator areas so a noisy rate is not read as a finding.
- **A construct-balanced variant of the Service Need Score.** Three of its nine components
  sit in one construct (see section 13), so the poverty construct carries 3/9 of the weight.
  The rankings are probably robust to this, but showing it beats asserting it.

### Three things to be ready to say

1. **Why 55 rows and not 56.** The city row is a benchmark, not a community.
2. **Why the health layer is 2018.** Nothing after 2019 contains a health outcome at all.
3. **Why the score is a screen, not a measurement.** Equal weighting is a choice, and
   percentile ranks are relative within a year — they cannot show improvement over time.
